In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from torchvision.datasets import MNIST
from sklearn.model_selection import train_test_split

data = MNIST('mnist', download=True)
images, labels = data.data, data.targets
X = images.view(-1, 28*28).float() / 255
X_train, X_valid, y_train, y_valid = train_test_split(X, labels, test_size=0.1, random_state=0)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 341kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.44MB/s]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
preds = lr.predict(X_valid)
accuracy_score(y_valid, preds)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9273333333333333

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest model
rf = RandomForestClassifier(random_state=0)
rf.fit(X_train, y_train)

# Predict and evaluate the accuracy
rf_preds = rf.predict(X_valid)
rf_accuracy = accuracy_score(y_valid, rf_preds)
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")

Random Forest Accuracy: 0.9727


In [ ]:
import torch
from torch import nn

class TorchLogReg(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

Définition de p (one-hot encoded)

In [ ]:
# Manuellement :
p_train = torch.zeros((len(y_train), 10), dtype=torch.float32)
p_train[torch.arange(len(y_train)), y_train] = 1

# Avec torch:
p_train2 = nn.functional.one_hot(y_train, num_classes=10).float()

(p_train == p_train2).all()

tensor(True)

In [ ]:
torch_log_reg = TorchLogReg(784, 10).cuda()
loss = nn.MSELoss(reduction='sum')
optimizer = torch.optim.SGD(torch_log_reg.parameters(), lr=0.01)
X_train = X_train.to('cuda')
X_valid = X_valid.to('cuda')
p_train = p_train.to('cuda')

for epoch in range(1000):
    optimizer.zero_grad()
    y_pred = torch_log_reg(X_train)
    p_pred = nn.functional.softmax(y_pred, dim=1)
    # loss_result = loss(p_pred, p_train)
    loss_result = torch.sum((p_pred - p_train) ** 2)
    loss_result.backward()
    optimizer.step()
    if epoch % 50 == 0:
        with torch.no_grad():
            pred_valid = torch_log_reg(X_valid)
        prediction = torch.argmax(pred_valid.to('cpu'), dim=1)
        accuracy = accuracy_score(y_valid, prediction)
        print(f'{accuracy:.2%}')

53.87%
80.28%
82.57%
81.73%
83.00%
89.48%
90.10%
91.50%
91.03%
86.88%
91.72%
92.15%
92.02%
90.60%
91.30%
92.20%
92.40%
92.42%
92.23%
92.17%


## nn.Linear à la main
Implémenter à la main nn.Linear, en définissant les poids dans `__init__` et le produit scalaire + biais dans `forward`. À noter :

- Les paramètres doivent être définis avec `nn.Parameter` ; par exemple, similaire à `self.A = nn.Parameter(torch.zeros(dim1, dim2))` (mais probablement initialisés avec des valeurs aléatoires, pas zéro). Sinon, ils seraient des constantes et pas mises à jours par `backward`.
- Le produit de tenseurs en PyTorch est appelé par `torch.matmul` ou bien avec `@`.

In [ ]:
class MyLinear(nn.Module):
    def __init__(self, dim1, dim2):
        super().__init__()
        # Initialize A as Normal b as zero
        self.A = nn.Parameter(torch.randn(dim1, dim2))
        self.b = nn.Parameter(torch.zeros(dim2))

    def forward(self, X):
        # X.shape: (n, dim1)
        return torch.matmul(X, self.A) + self.b

# Pour tester :
torch_log_reg = MyLinear(28*28, 10).cuda()
# Et lancer le même code que ci-dessus. Cela devrait donner les mêmes résultats

In [ ]:
loss = nn.CrossEntropyLoss(reduction='sum')
optimizer = torch.optim.SGD(torch_log_reg.parameters(), lr=0.0001)
X_train = X_train.to('cuda')
X_valid = X_valid.to('cuda')
y_train = y_train.to('cuda')

for epoch in range(1000):
    optimizer.zero_grad()
    y_pred = torch_log_reg(X_train)
    loss_result = loss(y_pred, y_train)
    loss_result.backward()
    optimizer.step()
    if epoch % 50 == 0:
        with torch.no_grad():
            pred_valid = torch_log_reg(X_valid)
        prediction = torch.argmax(pred_valid.to('cpu'), dim=1)
        accuracy = accuracy_score(y_valid, prediction)
        print(f'{accuracy:.2%}')

17.25%
86.62%
86.80%
89.62%
76.92%
88.60%
89.40%
90.97%
91.10%
91.20%
91.15%
91.27%
90.62%
91.55%
91.30%
91.60%
91.73%
64.72%
91.78%
91.70%


## SGD
Créer une classe SGD qui fait le même que `torch.optim.SGD`, et vérifier qu'il marche en remplaçant `torch.optim.SGD` par cette fonction dans l'entraînement de la régression logistique. Ceci doit avoir trois méthodes :

- `__init__(self, params, lr)` qui mémorise ces paramètres dans des attributes
- `step(self, *args, **kwargs)` qui pour tout paramètre `param` mémorisé, modifie `param.data` en soustrayant `param.grad * lr`
- `zero_grad(self, *args, **kwargs)` qui pour tout paramètre `param` appelle `.grad.zero_()` comme ci-dessus.

In [ ]:
g = torch_log_reg.parameters()

In [ ]:
for x in g:
    print(x)

Parameter containing:
tensor([[-0.4189,  1.0875, -1.6740,  ...,  0.3466, -2.0624, -1.5636],
        [ 0.3578, -1.4509, -0.9401,  ...,  1.7999, -0.4297,  0.5740],
        [ 0.6700, -1.3482,  1.8667,  ...,  1.5257,  0.2313,  0.5969],
        ...,
        [ 0.8091,  1.4263, -0.8672,  ...,  1.0125, -0.1434,  3.1087],
        [ 0.3853, -1.0851,  0.9117,  ..., -1.1749, -0.2713, -0.1186],
        [ 0.9053,  0.8276, -0.1133,  ...,  0.7611,  0.2847,  0.2148]],
       device='cuda:0', requires_grad=True)
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       requires_grad=True)


In [ ]:
for x in g:
    print(x)

In [ ]:
g = list(torch_log_reg.parameters())

In [ ]:
for x in g:
    print(x)

Parameter containing:
tensor([[-0.4189,  1.0875, -1.6740,  ...,  0.3466, -2.0624, -1.5636],
        [ 0.3578, -1.4509, -0.9401,  ...,  1.7999, -0.4297,  0.5740],
        [ 0.6700, -1.3482,  1.8667,  ...,  1.5257,  0.2313,  0.5969],
        ...,
        [ 0.8091,  1.4263, -0.8672,  ...,  1.0125, -0.1434,  3.1087],
        [ 0.3853, -1.0851,  0.9117,  ..., -1.1749, -0.2713, -0.1186],
        [ 0.9053,  0.8276, -0.1133,  ...,  0.7611,  0.2847,  0.2148]],
       device='cuda:0', requires_grad=True)
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       requires_grad=True)


In [ ]:
for x in g:
    print(x)

Parameter containing:
tensor([[-0.4189,  1.0875, -1.6740,  ...,  0.3466, -2.0624, -1.5636],
        [ 0.3578, -1.4509, -0.9401,  ...,  1.7999, -0.4297,  0.5740],
        [ 0.6700, -1.3482,  1.8667,  ...,  1.5257,  0.2313,  0.5969],
        ...,
        [ 0.8091,  1.4263, -0.8672,  ...,  1.0125, -0.1434,  3.1087],
        [ 0.3853, -1.0851,  0.9117,  ..., -1.1749, -0.2713, -0.1186],
        [ 0.9053,  0.8276, -0.1133,  ...,  0.7611,  0.2847,  0.2148]],
       device='cuda:0', requires_grad=True)
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       requires_grad=True)


In [ ]:
class MySGD():
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    def step(self):
        for param in self.params:
            param.data -= self.lr * param.grad

    def zero_grad(self):
        for param in self.params:
            param.grad = None
            # if param.grad is not None:
            #     param.grad.zero_()

In [ ]:
torch_log_reg = MyLinear(28*28, 10).cuda()
loss = nn.CrossEntropyLoss(reduction='sum')
optimizer = MySGD(torch_log_reg.parameters(), lr=0.0001)
X_train = X_train.to('cuda')
X_valid = X_valid.to('cuda')
y_train = y_train.to('cuda')

for epoch in range(1000):
    optimizer.zero_grad()
    y_pred = torch_log_reg(X_train)
    loss_result = loss(y_pred, y_train)
    loss_result.backward()
    optimizer.step()
    if epoch % 50 == 0:
        with torch.no_grad():
            pred_valid = torch_log_reg(X_valid)
        prediction = torch.argmax(pred_valid.to('cpu'), dim=1)
        accuracy = accuracy_score(y_valid, prediction)
        print(f'{accuracy:.2%}')

31.30%
85.27%
88.23%
89.05%
89.35%
90.23%
90.37%
90.67%
90.87%
91.05%
90.80%
91.40%
90.92%
91.00%
91.38%
71.33%
90.82%
91.77%
91.78%
91.75%
